# Judge period strings with Claude

Sends each `data/judge-input/batch-*.jsonl` without an answer yet to Claude through the Message
Batches API and saves the reply as `data/judge-output/<batch>.jsonl` for `period_parsers.ipynb`.
The system prompt and the per-batch instruction come from `period_judge_prompt.md`. Needs the
`anthropic` package and an API key, read from `ANTHROPIC_API_KEY` or asked for. `LIMIT` trials a
few batches; re-running fills whatever is still missing.

In [ ]:
%pip install anthropic

In [17]:
import getpass
import json
import os
import time
from pathlib import Path

import anthropic

MODEL = "claude-sonnet-5"
INPUT_DIR = Path("data/judge-input")
OUTPUT_DIR = Path("data/judge-output")
LIMIT = 3

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY") or getpass.getpass("Anthropic API key: "))

In [18]:
system_text, _, user_part = Path("period_judge_prompt.md").read_text(encoding="utf-8").partition("# User message, per batch")
user_template = user_part.split("```")[1].strip().splitlines()[0]  # the instruction line; the batch follows it

system = [{"type": "text", "text": system_text.removeprefix("# System prompt").strip(), "cache_control": {"type": "ephemeral"}}]
print(user_template)

Here are {n} strings. Return one JSON array per line for each id, in order.


In [20]:
pending = [f for f in sorted(INPUT_DIR.glob("batch-*.jsonl")) if not (OUTPUT_DIR / f.name).exists()][:LIMIT]

requests = []
for f in pending:
    lines = f.read_text(encoding="utf-8").strip().splitlines()
    requests.append({
        "custom_id": f.stem,
        "params": {
            "model": MODEL,
            "max_tokens": 8000,
            "system": system,
            "output_config": {"effort": "low"},
            "messages": [{"role": "user", "content": user_template.format(n=len(lines)) + "\n\n" + "\n".join(lines)}],
        },
    })

batch = client.messages.batches.create(requests=requests)
print(f"submitted {len(requests)} requests as batch {batch.id}")

submitted 3 requests as batch msgbatch_01K1SwNby2sEyR5ZUQmtcs3W


In [ ]:
while (batch := client.messages.batches.retrieve(batch.id)).processing_status != "ended":
    print(f"{batch.request_counts.processing} still processing")
    time.sleep(60)
print(batch.request_counts)

3 still processing
3 still processing


In [16]:
OUTPUT_DIR.mkdir(exist_ok=True)
usage = {"input": 0, "cached": 0, "output": 0}
for result in client.messages.batches.results(batch.id):
    if result.result.type != "succeeded":
        print(f"{result.custom_id}: {result.result.type}")
        continue
    message = result.result.message
    text = "".join(block.text for block in message.content if block.type == "text")
    (OUTPUT_DIR / f"{result.custom_id}.jsonl").write_text(text.strip() + "\n", encoding="utf-8")
    usage["input"] += message.usage.input_tokens
    usage["cached"] += message.usage.cache_read_input_tokens or 0
    usage["output"] += message.usage.output_tokens

print(f"{len(list(OUTPUT_DIR.glob('batch-*.jsonl')))} batches now judged; tokens this run: {usage}")

3 batches now judged; tokens this run: {'input': 10755, 'cached': 6958, 'output': 13462}
